# C 노트북 · ML 예측, BG/NBD, 가설 H3

**담당:** C (이름: 이정민) · **중간 점검:** 11/29 · **최종 산출:** ML 섹션, BG/NBD(도전 파트), H3 검정, 1장 도입 초안

**이 노트북에서 할 일**
1. S1로 학습, S2로 검증, S3로 최종 테스트 (아래 로지스틱 기준 모델 구현 있음)
2. 모델을 로지스틱 → 랜덤 포레스트/그래디언트 부스팅으로 확장, 기준선(`baseline_recency_score`)보다 나은지 H3 검정 (AUC·PR-AUC, 부트스트랩 신뢰구간, S3 홀드아웃)
3. BG/NBD + Gamma-Gamma (도전 파트): **가정 점검** 관점으로 쓴다. 이 데이터는 거래처(도매) 구매라 구매 간격이 포아송보다 규칙적이다 (CV 중앙값 0.75)
4. 모든 점수는 `ru.to_score_frame`으로 낸다

In [1]:
import sys; sys.path.insert(0, "..")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import rfm_utils as ru

clean, report = ru.clean_transactions(ru.load_raw())
inv = ru.invoice_table(clean)
datasets = {name: ru.make_dataset(inv, snap) for name, snap in ru.SNAPSHOTS.items()}
for name, snap in ru.SNAPSHOTS.items():
    ru.assert_no_leakage(inv, datasets[name], snap)   # 새 데이터셋을 만들 때마다 호출
{k: len(v) for k, v in datasets.items()}   # 기준일별 고객 수

{'S1': 4537, 'S2': 4933, 'S3': 5249}

## 1. 로지스틱 기준 모델
입력 열은 기준일 이전 정보만 쓴 `R, F, M, avg_order_value, T`. 금액은 치우쳐 있어 `log1p`를 쓴다.

In [2]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

FEATURES = ["R", "F", "M", "avg_order_value", "T"]
def X_of(d): return np.log1p(d[FEATURES])

tr, va, te = datasets["S1"], datasets["S2"], datasets["S3"]
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=ru.SEED))
model.fit(X_of(tr), tr["repurchase"])

rows = []
for name, d in [("S2 (validation)", va), ("S3 (test)", te)]:
    for method, score in [("logistic", model.predict_proba(X_of(d))[:, 1]),
                          ("recency_only", ru.baseline_recency_score(d).values)]:
        rows.append({"split": name, "method": method, **ru.evaluate_scores(d["repurchase"], score)})
pd.DataFrame(rows).round(3)

/private/tmp/claude-501/-Users-changmin-Desktop-AI---------final-project/30771d6c-1b9f-4e24-ab9f-31ed5e2310af/scratchpad/venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/private/tmp/claude-501/-Users-changmin-Desktop-AI---------final-project/30771d6c-1b9f-4e24-ab9f-31ed5e2310af/scratchpad/venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/private/tmp/claude-501/-Users-changmin-Desktop-AI---------final-project/30771d6c-1b9f-4e24-ab9f-31ed5e2310af/scratchpad/venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/private/tmp/claude-501/-Users-changmin-Desktop-AI---------final-project/30771d6c-1b9f-4e24-ab9f-31ed5e2310af/scratchpad/venv/lib/p

,split,method,auc,pr_auc,base_rate,capture@10,precision@10,lift@10
0,S2 (validation),logistic,0.811,0.704,0.329,0.263,0.864,2.630
1,S2 (validation),recency_only,0.775,0.587,0.329,0.205,0.673,2.049
2,S3 (test),logistic,0.786,0.743,0.427,0.206,0.878,2.057
3,S3 (test),recency_only,0.762,0.672,0.427,0.178,0.760,1.780


**TODO (C):** 랜덤 포레스트·그래디언트 부스팅 추가, 하이퍼파라미터는 S2로만 고르고 S3는 마지막에 한 번만 본다. H3: AUC 차이의 부트스트랩 95% 신뢰구간 (고객 단위 재표집).

## 2. 공통 점수 형식으로 내기

In [3]:
sf = ru.to_score_frame(te.index, model.predict_proba(X_of(te))[:, 1], ru.SNAPSHOTS["S3"], "logistic")
ru.validate_score_frame(sf, te)
sf.head()

/private/tmp/claude-501/-Users-changmin-Desktop-AI---------final-project/30771d6c-1b9f-4e24-ab9f-31ed5e2310af/scratchpad/venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/private/tmp/claude-501/-Users-changmin-Desktop-AI---------final-project/30771d6c-1b9f-4e24-ab9f-31ed5e2310af/scratchpad/venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/private/tmp/claude-501/-Users-changmin-Desktop-AI---------final-project/30771d6c-1b9f-4e24-ab9f-31ed5e2310af/scratchpad/venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


,customer_id,snapshot,method,score
0,12346,2011-09-01,logistic,0.805052
1,12347,2011-09-01,logistic,0.631783
2,12348,2011-09-01,logistic,0.363969
3,12349,2011-09-01,logistic,0.282085
4,12350,2011-09-01,logistic,0.126402


## 3. BG/NBD (도전 파트) 준비

`build_features`가 `recency_bgnbd`(마지막 구매일 − 첫 구매일), `frequency_bgnbd`(반복 구매 횟수), `T`, `repeat_avg_amount`를 이미 만들어 준다. 주의: BG/NBD의 recency는 규칙 기반 RFM의 R과 **정의가 다르다.**

라이브러리 호환성부터 확인한다 (`lifetimes`는 관리가 멈췄을 수 있다. 안 되면 PyMC-Marketing).

In [4]:
try:
    import lifetimes
    print("lifetimes", lifetimes.__version__)
except Exception as e:
    print("lifetimes 사용 불가:", type(e).__name__, e)
try:
    import pymc_marketing
    print("pymc-marketing", pymc_marketing.__version__)
except Exception as e:
    print("pymc-marketing 사용 불가:", type(e).__name__)
datasets["S1"][["recency_bgnbd", "frequency_bgnbd", "T", "repeat_avg_amount"]].describe().round(1)

lifetimes 사용 불가: ModuleNotFoundError No module named 'lifetimes'
pymc-marketing 사용 불가: ModuleNotFoundError


,recency_bgnbd,frequency_bgnbd,T,repeat_avg_amount
count,4537.0,4537.0,4537.0,4537.0
mean,153.4,3.8,291.8,251.4
std,152.9,9.0,130.5,409.4
min,0.0,0.0,0.0,0.0
25%,0.0,0.0,166.0,0.0
50%,118.0,1.0,320.0,180.6
75%,294.0,4.0,405.0,339.8
max,453.0,219.0,454.0,8416.4


## 다음 단계 (C)
- [ ] 랜덤 포레스트/그래디언트 부스팅, 검증 점수 표
- [ ] H3 검정(부트스트랩 CI)과 S3 최종 결과
- [ ] 라이브러리 확정 후 `requirements.txt`에 버전 추가
- [ ] BG/NBD 적합, 기대 구매 횟수를 점수로 내서 같은 표에 올리기
- [ ] 포아송 가정 점검 (구매 간격 CV 분포)을 그림으로
- [ ] 1장 도입 초안 ("RFM은 정말 미래를 예측하는가?")
- [ ] 막히면 BG/NBD는 줄이고 ML을 먼저 끝낸다 (기획안 9장 "지연이 생기면")

## 4. 결과 저장 (A에게 넘기기)
검증(S2)과 테스트(S3) 점수를 `outputs/scores_logistic.csv`로 저장하고 커밋한다. 모델을 개선할 때마다 다시 실행해 덮어쓴다. 새 방법(랜덤 포레스트, BG/NBD)은 `method` 이름을 바꿔 `scores_<method>.csv`로 따로 저장한다.

In [5]:
frames = [ru.to_score_frame(d.index, model.predict_proba(X_of(d))[:, 1], ru.SNAPSHOTS[name], "logistic")
          for name, d in [("S2", va), ("S3", te)]]
print(ru.save_score_frame(pd.concat(frames, ignore_index=True), "logistic"))

/Users/changmin/Desktop/AI데이터분석및실습/final project/outputs/scores_logistic.csv


/private/tmp/claude-501/-Users-changmin-Desktop-AI---------final-project/30771d6c-1b9f-4e24-ab9f-31ed5e2310af/scratchpad/venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/private/tmp/claude-501/-Users-changmin-Desktop-AI---------final-project/30771d6c-1b9f-4e24-ab9f-31ed5e2310af/scratchpad/venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/private/tmp/claude-501/-Users-changmin-Desktop-AI---------final-project/30771d6c-1b9f-4e24-ab9f-31ed5e2310af/scratchpad/venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/private/tmp/claude-501/-Users-changmin-Desktop-AI---------final-project/30771d6c-1b9f-4e24-ab9f-31ed5e2310af/scratchpad/venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/pr